In [32]:
import ollama
import json
import math
import argparse
import os
import openpyxl


In [33]:
# --- Helper Functions ---
def dot_product(v1, v2):
    return sum(a * b for a, b in zip(v1, v2))

def magnitude(v):
    return math.sqrt(sum(a * a for a in v))

def cosine_similarity(v1, v2):
    return dot_product(v1, v2) / (magnitude(v1) * magnitude(v2))

In [34]:
# --- Convert a raw row dict into a natural language summary ---
def row_to_summary(raw):
    """
    Turns structured fields into a descriptive sentence the embedding
    model can reason about semantically.
    """
    parts = []

    vehicle = raw.get("Vehicle Type", "Vehicle")
    duration = raw.get("Duration (Mins)")
    fee = raw.get("Total Fee", "unknown fee")
    payment = raw.get("Payment Method", "unknown method")
    entry = raw.get("Entry Time", "")
    exit_ = raw.get("Exit Time", "")
    tid = raw.get("Transaction ID", "")

    # Duration in human terms
    if duration:
        try:
            mins = int(duration)
            hours = mins // 60
            remaining = mins % 60
            if hours > 0:
                duration_text = f"{hours} hour{'s' if hours > 1 else ''} and {remaining} minutes" if remaining else f"{hours} hour{'s' if hours > 1 else ''}"
            else:
                duration_text = f"{mins} minutes"

            # Tag it
            if mins >= 300:
                duration_text += " (very long stay)"
            elif mins >= 120:
                duration_text += " (long stay)"
            elif mins <= 30:
                duration_text += " (short stay)"
        except:
            duration_text = f"{duration} minutes"
    else:
        duration_text = "unknown duration"

    # Fee in human terms
    fee_text = fee
    if fee:
        try:
            amount = float(str(fee).replace("$", "").replace(",", ""))
            if amount == 0:
                fee_text = "free (validated or exempt)"
            elif amount >= 20:
                fee_text = f"${amount:.2f} (expensive)"
            elif amount >= 10:
                fee_text = f"${amount:.2f} (moderate cost)"
            else:
                fee_text = f"${amount:.2f} (cheap)"
        except:
            fee_text = fee

    parts.append(f"Transaction {tid}: A {vehicle} entered at {entry} and exited at {exit_}.")
    parts.append(f"The visit lasted {duration_text}.")
    parts.append(f"Total fee was {fee_text}, paid by {payment}.")

    return " ".join(parts)

In [35]:
filepath = "../test_data/data.xlsx"


def load_excel(filepath):
    wb = openpyxl.load_workbook(filepath)
    chunks = []
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        rows = list(ws.iter_rows(values_only=True))
        if not rows:
            continue
        headers = [str(h) if h else "" for h in rows[0]]
        for row in rows[1:]:
            if not any(row):
                continue
            raw = {headers[i]: str(val) for i, val in enumerate(row) if val is not None}
            original = " | ".join([f"{headers[i]}: {str(val)}" for i, val in enumerate(row) if val is not None])
            summary = row_to_summary(raw)   # <-- richer text for embedding
            chunks.append({
                "text": original,           # shown to user
                "summary": summary,         # used for embedding
                "raw": raw,
                "source": f"{filepath} [{sheet_name}]"
            })
    return chunks

In [36]:
# --- Embed and Save ---

# Global variable to store embedded data
embedded_data = None

def embed_excel(excel_files):
    global embedded_data
    data_to_store = []
    for file_path in excel_files:
        if not os.path.exists(file_path):
            print(f"Warning: {file_path} not found, skipping...")
            continue
        if not file_path.endswith(('.xlsx', '.xls')):
            print(f"Warning: {file_path} is not an Excel file, skipping...")
            continue
        print(f"Processing {file_path}...")
        chunks = load_excel(file_path)
        for chunk in chunks:
            print(f"  Embedding: {chunk['summary'][:50]}...")
            response = ollama.embed(model='nomic-embed-text', input=chunk["summary"])
            vector = response['embeddings'][0]
            data_to_store.append({
                "text": chunk["text"],
                "summary": chunk["summary"],
                "raw": chunk["raw"],
                "source": chunk["source"],
                "vector": vector
            })
        print(f"  → {len(chunks)} rows embedded from {file_path}")
    
    with open("excel_vectors.json", "w") as f:
        json.dump(data_to_store, f, indent=2)
    print(f"\nDone! Saved {len(data_to_store)} entries to excel_vectors.json")

    # Store in global variable
    embedded_data = data_to_store
    return data_to_store


# Call with the test data file
embedded_data = embed_excel(["../test_data/data.xlsx"])

Processing ../test_data/data.xlsx...
  Embedding: Transaction CP-1001: A SUV entered at 08:15:22 and...
  Embedding: Transaction CP-1002: A Sedan entered at 08:30:45 a...
  Embedding: Transaction CP-1003: A Electric entered at 09:05:1...
  Embedding: Transaction CP-1004: A Truck entered at 10:12:33 a...
  Embedding: Transaction CP-1005: A Sedan entered at 11:45:00 a...
  Embedding: Transaction CP-1006: A SUV entered at 12:20:15 and...
  Embedding: Transaction CP-1007: A Motorcycle entered at 13:10...
  Embedding: Transaction CP-1008: A Sedan entered at 14:00:00 a...
  → 8 rows embedded from ../test_data/data.xlsx

Done! Saved 8 entries to excel_vectors.json


In [37]:
# --- Exact match filter ---
def apply_filter(database, filter_str):
    filters = [f.strip() for f in filter_str.split(",")]
    filtered = []
    for entry in database:
        match = True
        for f in filters:
            if ":" not in f:
                continue
            field, value = f.split(":", 1)
            field, value = field.strip().lower(), value.strip().lower()
            raw = {k.lower(): v.lower() for k, v in entry["raw"].items()}
            if not any(field in k and value in v for k, v in raw.items()):
                match = False
                break
        if match:
            filtered.append(entry)
    return filtered

In [39]:
# --- Search ---
def search(query, top_k=3, filter_str=None, show_summary=False):
    global embedded_data
    
    if embedded_data is None:
        print("No embedded data. Run embed_excel() first.")
        return

    database = embedded_data.copy()

    if filter_str:
        database = apply_filter(database, filter_str)
        print(f"Filter applied: {len(database)} rows matched")

    if not database:
        print("No results after filtering.")
        return

    response = ollama.embed(model='nomic-embed-text', input=query)
    query_vector = response['embeddings'][0]

    results = []
    for entry in database:
        score = cosine_similarity(query_vector, entry["vector"])
        results.append((score, entry["source"], entry["text"], entry.get("summary", "")))

    results.sort(reverse=True)
    print(f"\nSearch results for: '{query}'\n{'='*50}")
    for i, (score, source, text, summary) in enumerate(results[:top_k], 1):
        print(f"\nResult #{i} (score: {score:.4f})")
        print(f"Source: {source}")
        for field in text.split(" | "):
            print(f"  {field}")
        if show_summary:
            print(f"  [Embedded as]: {summary}")
        print("---")


search("long parking stay with credit card payment", filter_str="Vehicle Type: SUV, Payment Method: Credit Card", show_summary=True)

Filter applied: 2 rows matched

Search results for: 'long parking stay with credit card payment'

Result #1 (score: 0.5259)
Source: ../test_data/data.xlsx [Sheet1]
  Transaction ID: CP-1006
  Entry Time: 12:20:15
  Exit Time: 15:40:10
  Duration (Mins): 200
  License Plate (Masked): JKL-***
  Vehicle Type: SUV
  Total Fee: $16.00
  Payment Method: Credit Card
  [Embedded as]: Transaction CP-1006: A SUV entered at 12:20:15 and exited at 15:40:10. The visit lasted 3 hours and 20 minutes (long stay). Total fee was $16.00 (moderate cost), paid by Credit Card.
---

Result #2 (score: 0.5197)
Source: ../test_data/data.xlsx [Sheet1]
  Transaction ID: CP-1001
  Entry Time: 08:15:22
  Exit Time: 10:45:10
  Duration (Mins): 150
  License Plate (Masked): ABC-***
  Vehicle Type: SUV
  Total Fee: $12.00
  Payment Method: Credit Card
  [Embedded as]: Transaction CP-1001: A SUV entered at 08:15:22 and exited at 10:45:10. The visit lasted 2 hours and 30 minutes (long stay). Total fee was $12.00 (modera